<a href="https://colab.research.google.com/github/kjwang24/role-boundary-info-transmission/blob/main/role_boundary_info_transmission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# `<think>` to `<assistant>` Transmission

**Is answer-relevant information lost across the boundary
between the end of the `<think>` block and the start of the assistant answer?**

Two possible modes of loss:
- **Representational:** is the answer *decodable* from the residual stream at the end of `<think>`, and does that decodability *survive* to the first answer position? → test with **logit lens, linear probes**
- **Routing:** does the answer causally *read* the think/boundary information via attention? → test with **attention knockout**

Tests and what they indicate:
| | knockout hurts | knockout no effect |
|---|---|---|
| **decodable at handoff** | transmitted & used | **lost at the handoff** |
| **not decodable** | nonlinear / late | never consolidated |

This colab tests DeepSeek-R1-Distill-Qwen-1.5B on a prompt where the answer only appears inside the `<think>` block, so it has to cross the `<think>`/`<assistant>` boundary to be produced.

## 1. Setup / install

In [2]:
!pip install -q transformer_lens scikit-learn

In [3]:
import os, gc, json, functools, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from google.colab import userdata
from huggingface_hub import login

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
torch.set_grad_enabled(False)          # pure inference, probes run on extracted features
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
print(f"device={device}  dtype={dtype}")
if device != "cuda":
    print("WARNING: no GPU found, this will be slow. On Colab pick Runtime > Change runtime type > GPU of choice")

device=cuda  dtype=torch.float16


## 2. Model loading

DeepSeek-R1-Distill-Qwen-1.5B isn't in the TransformerLens registry, but it has the same architecture as Qwen2.5-1.5B-Instruct. We ask TransformerLens to prepare Qwen2's skeleton and then fill it with DeepSeek's weights, downloaded from HuggingFace. We keep the DeepSeek tokenizer.

In [ ]:
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
TL_ALIAS = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, low_cpu_mem_usage=True)
hf_token = userdata.get('HF_READ_PUBLIC')

if hf_token:
    login(hf_token)
else:
    raise ValueError("HF token misconfigured, please check Secrets manager or HF account")

# make sure qwen2 and deepseek weights match
c = hf_model.config
expected = dict(num_hidden_layers=28, hidden_size=1536, num_attention_heads=12,
                num_key_value_heads=2, intermediate_size=8960, vocab_size=151936)
for k, v in expected.items():
    assert getattr(c, k) == v, f"config mismatch on {k}: got {getattr(c, k)}, expected {v}"
print("config parity OK:", expected)

model = HookedTransformer.from_pretrained_no_processing(
    TL_ALIAS, hf_model=hf_model, tokenizer=tokenizer, device=device,
    dtype="float16" if device == "cuda" else "float32",
)
model.eval()
print(f"loaded: n_layers={model.cfg.n_layers}  d_model={model.cfg.d_model}  d_vocab={model.cfg.d_vocab}")

# make sure tl reproduces hf logits (guards the alias/dtype/softcap-free path)
hf_model.to(device)
probe_ids = tokenizer("The capital of France is", return_tensors="pt").input_ids.to(device)
hf_last = hf_model(probe_ids).logits[0, -1].float()
tl_last = model(probe_ids)[0, -1].float()
print("HF/TL argmax match:", int(hf_last.argmax()) == int(tl_last.argmax()),
      " max|delta|:", round((hf_last - tl_last).abs().max().item(), 3))

del hf_model; gc.collect()
if device == "cuda": torch.cuda.empty_cache()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

config parity OK: {'num_hidden_layers': 28, 'hidden_size': 1536, 'num_attention_heads': 12, 'num_key_value_heads': 2, 'intermediate_size': 8960, 'vocab_size': 151936}


## 3. Dataset and clean/corrupt forward passes

A single fixed template with a varying **secret value**. The value is planted **only** inside the `<think>` block, there are no hints to it in the question, so it must be transmitted across the
boundary. Items and values are constrained to single tokens so every sequence is token-aligned (same length, same position for every structural token). The **minimal pair** for each example swaps only the secret value, giving a clean causal contrast.

In [1]:
QUESTION      = "State the secret {item}."
THINK_BODY    = " The secret {item} is {value}."     # the value appears ONLY here
THINK_CLOSE   = "\n</think>\n\n"
ANSWER_PREFIX = "The secret {item} is"

ITEM_CANDS  = ["number","word","code","color","animal","name","object","symbol","token","letter"]
VALUE_CANDS = ["red","blue","green","gold","seven","three","dog","cat","tree","star","moon","fish",
               "bird","rose","king","queen","salt","milk","fire","rain","snow","wind","rock","sand"]

def one_token_id(word):
    # single leading-space token id, or none if the word is not a single token
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    return ids[0] if len(ids) == 1 else None

items  = [w for w in ITEM_CANDS if one_token_id(w) is not None][:10]
values = sorted({w for w in VALUE_CANDS if one_token_id(w) is not None})[:16]
val_id = {w: one_token_id(w) for w in values}
print(f"{len(items)} single-token items, {len(values)} single-token values")

def build_prompt(item, value):
    msgs   = [{"role": "user", "content": QUESTION.format(item=item)}]
    prefix = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
    if "<think>" not in tokenizer.decode(prefix):        # ensure the think block is opened
        prefix = prefix + tokenizer.encode("<think>\n", add_special_tokens=False)
    tb = tokenizer.encode(THINK_BODY.format(item=item, value=value), add_special_tokens=False)
    tc = tokenizer.encode(THINK_CLOSE, add_special_tokens=False)
    ap = tokenizer.encode(ANSWER_PREFIX.format(item=item), add_special_tokens=False)
    toks = prefix + tb + tc + ap
    pos = dict(think_start=len(prefix), think_end=len(prefix)+len(tb),
               close_start=len(prefix)+len(tb), close_end=len(prefix)+len(tb)+len(tc),
               answer=len(toks)-1)
    pos["value_think"] = pos["think_start"] + tb.index(val_id[value])  # where the value sits in <think>
    return toks, pos, val_id[value]

NameError: name 'tokenizer' is not defined

In [ ]:
rng = random.Random(SEED)
examples, ref_len, ref_pos = [], None, None
for item in items:
    for value in values:
        toks, pos, aid = build_prompt(item, value)
        if ref_len is None:
            ref_len, ref_pos = len(toks), pos
        if len(toks) != ref_len or pos != ref_pos:
            continue                                     # keep only fully-aligned examples
        value2 = rng.choice([v for v in values if v != value])
        toks2, pos2, aid2 = build_prompt(item, value2)
        if len(toks2) != ref_len or pos2 != ref_pos:
            continue
        examples.append(dict(item=item, value=value, value2=value2,
                             clean=toks, corrupt=toks2, ans=aid, ans2=aid2))

POS = ref_pos                                            # positions are identical across examples
print(f"{len(examples)} aligned examples, seq_len={ref_len}")
print("loci:", {k: POS[k] for k in ["value_think","close_end","answer"]})
print("example clean prompt:\n", tokenizer.decode(examples[0]["clean"]))

In [ ]:
clean_ids   = torch.tensor([e["clean"]   for e in examples], device=device)
corrupt_ids = torch.tensor([e["corrupt"] for e in examples], device=device)
ans_ids     = torch.tensor([e["ans"]  for e in examples])
ans2_ids    = torch.tensor([e["ans2"] for e in examples])
ap = POS["answer"]
N  = len(examples)
idx = torch.arange(N)

def answer_logits(ids, batch=48):
    out = []
    for i in range(0, len(ids), batch):
        out.append(model(ids[i:i+batch])[:, ap, :].float().cpu())
    return torch.cat(out)

clean_al = answer_logits(clean_ids)
corr_al  = answer_logits(corrupt_ids)
clean_probs = clean_al.softmax(-1)

def logit_diff(al):                                      # secret value minus minimal-pair value
    return al[idx, ans_ids] - al[idx, ans2_ids]

clean_ld, corr_ld = logit_diff(clean_al), logit_diff(corr_al)
top1 = (clean_al.argmax(-1) == ans_ids).float().mean().item()
print(f"clean top-1 == secret value: {top1:.1%}")
print(f"mean logit-diff  clean={clean_ld.mean():.2f}  corrupt={corr_ld.mean():.2f}")
assert clean_ld.mean() > corr_ld.mean(), "corruption did not flip the answer — the task leaks the value"

## 4. Logit lens

Project each layer's residual (at three positions) through `ln_final`, then unembed and read the rank of the
secret-value token. Low rank → the value is decodable there. We watch the trajectory across
`value_think` (where the token is) → `boundary` (`</think>`) → `answer` (destination).

In [ ]:
LOCI = {"value_think": POS["value_think"], "boundary": POS["close_end"] - 1, "answer": POS["answer"]}
resid_names = [f"blocks.{L}.hook_resid_post" for L in range(model.cfg.n_layers)]

def cache_resid(ids, loci, batch=48):
    store = {p: [] for p in loci}
    for i in range(0, len(ids), batch):
        _, cache = model.run_with_cache(ids[i:i+batch],
                                        names_filter=lambda n: n.endswith("resid_post"))
        stacked = torch.stack([cache[n] for n in resid_names], dim=1)   # [b, n_layers, seq, d]
        for name, p in loci.items():
            store[name].append(stacked[:, :, p, :].float().cpu())
        del cache, stacked
    return {p: torch.cat(v) for p, v in store.items()}

resid = cache_resid(clean_ids, LOCI)                    # {locus: [N, n_layers, d_model]}

def lens_metrics(resid_locus):
    n_layers = model.cfg.n_layers
    rank = np.zeros((N, n_layers)); prob = np.zeros((N, n_layers))
    aid = ans_ids.to(device)
    for L in range(n_layers):
        h = resid_locus[:, L, :].to(device).to(dtype)
        logits = model.unembed(model.ln_final(h)).float()             # [N, vocab]
        al = logits[idx, aid.cpu()] if device == "cpu" else logits[torch.arange(N, device=device), aid]
        rank[:, L] = (logits > al[:, None]).sum(-1).cpu().numpy()
        prob[:, L] = logits.softmax(-1)[torch.arange(N, device=logits.device), aid.to(logits.device)].cpu().numpy()
        del logits
    return rank.mean(0), prob.mean(0)

lens_rank, lens_prob = {}, {}
for loc in LOCI:
    lens_rank[loc], lens_prob[loc] = lens_metrics(resid[loc])
    print(f"{loc:12s} min mean-rank over layers: {lens_rank[loc].min():.1f}")

## 5. Linear probes

Can a linear map read the secret value from the residual at each position/layer? We expect `value_think` to be
near-perfect, so it serves as a control reference. The drop
from boundary to answer measures representational attenuation across the handoff.

In [ ]:
y = np.array([e["value"] for e in examples])
chance = 1.0 / len(set(y))

def probe_acc(resid_locus):
    accs = []
    for L in range(model.cfg.n_layers):
        X = resid_locus[:, L, :].numpy()
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        accs.append(cross_val_score(clf, X, y, cv=5).mean())
    return np.array(accs)

probe = {loc: probe_acc(resid[loc]) for loc in LOCI}
for loc in LOCI:
    print(f"{loc:12s} peak CV acc {probe[loc].max():.2f}  (chance {chance:.2f})")
print(f"boundary->answer peak-accuracy drop: {probe['boundary'].max() - probe['answer'].max():+.2f}")

## 6. Attention knockout

Zero the **answer** position's attention to a target key-region (renormalizing the row), one layer at a time, and measure the answer degradation (logit-diff drop and Kullback-Leibler divergence from the clean run). A large effect means the answer **causally reads** that region across the boundary. Targets: `think` content, `boundary` (`</think>`), and `reasoning_all` (both — the definitive routing cut). We also block all layers at
once, which is the only configuration that severs multi-hop routing.

check later, why is `<assistant>` not analyzed?

In [ ]:
think_keys    = list(range(POS["think_start"], POS["think_end"]))
boundary_keys = list(range(POS["close_start"], POS["close_end"]))
KO_TARGETS = {"think": think_keys, "boundary": boundary_keys,
              "reasoning_all": think_keys + boundary_keys}

def ko_hook(pattern, hook, q, keys):
    pattern[:, :, q, keys] = 0.0                         # cut answer-query -> region-key edges
    row = pattern[:, :, q, :]
    pattern[:, :, q, :] = row / row.sum(-1, keepdim=True).clamp_min(1e-9)   # renormalize the row
    return pattern

def kl_from_clean(al):
    p, q = clean_probs, al.softmax(-1)
    return (p * (p.clamp_min(1e-9).log() - q.clamp_min(1e-9).log())).sum(-1)

def run_ko(keys, layers, batch=48):
    out = []
    for i in range(0, len(clean_ids), batch):
        hooks = [(f"blocks.{L}.attn.hook_pattern", functools.partial(ko_hook, q=ap, keys=keys))
                 for L in layers]
        out.append(model.run_with_hooks(clean_ids[i:i+batch], fwd_hooks=hooks)[:, ap, :].float().cpu())
    return torch.cat(out)

ko_ldrop = {t: np.zeros(model.cfg.n_layers) for t in KO_TARGETS}   # per-layer effect
ko_kl    = {t: np.zeros(model.cfg.n_layers) for t in KO_TARGETS}
for t, keys in KO_TARGETS.items():
    for L in range(model.cfg.n_layers):
        al = run_ko(keys, [L])
        ko_ldrop[t][L] = (clean_ld - logit_diff(al)).mean().item()
        ko_kl[t][L]    = kl_from_clean(al).mean().item()
    print(f"per-layer knockout done: {t}")

ko_total = {}                                            # block ALL layers at once
for t, keys in KO_TARGETS.items():
    al = run_ko(keys, list(range(model.cfg.n_layers)))
    ko_total[t] = dict(ldrop=(clean_ld - logit_diff(al)).mean().item(),
                       kl=kl_from_clean(al).mean().item())
print("block-all-layers logit-diff drop:", {t: round(ko_total[t]["ldrop"], 2) for t in KO_TARGETS})

## 7. Synthesis + visualization

In [ ]:
layers = np.arange(model.cfg.n_layers)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))

for loc in LOCI: axes[0].plot(layers, lens_rank[loc], marker="o", ms=3, label=loc)
axes[0].set(title="Logit lens: rank of secret token (lower = decodable)",
            xlabel="layer", ylabel="mean rank", yscale="log"); axes[0].legend()

for loc in LOCI: axes[1].plot(layers, probe[loc], marker="o", ms=3, label=loc)
axes[1].axhline(chance, ls="--", c="gray", lw=1, label="chance")
axes[1].set(title="Linear probe: decode secret value", xlabel="layer",
            ylabel="CV accuracy", ylim=(0, 1.02)); axes[1].legend()

for t in KO_TARGETS: axes[2].plot(layers, ko_ldrop[t], marker="o", ms=3, label=t)
axes[2].set(title="Attention knockout: answer -/-> region", xlabel="layer blocked",
            ylabel="clean - knockout logit-diff"); axes[2].legend()

plt.tight_layout(); plt.savefig(f"{OUT_DIR}/summary.png", dpi=130, bbox_inches="tight"); plt.show()

## 8. Analysis

Combine the three readouts into the 2×2 verdict and write `outputs/analysis.txt` and `outputs/metrics.json`.

In [ ]:
cl = float(clean_ld.mean())
dec_boundary = bool(probe["boundary"].max() > 3 * chance)
dec_answer   = bool(probe["answer"].max()   > 3 * chance)
reads_all    = bool(ko_total["reasoning_all"]["ldrop"] > 0.5 * cl)
via_boundary = bool(ko_total["boundary"]["ldrop"] > ko_total["think"]["ldrop"])
b_ans_drop   = float(probe["boundary"].max() - probe["answer"].max())

if   dec_boundary and reads_all:         cell = "TRANSMITTED & USED (present at </think> and causally read)"
elif dec_boundary and not reads_all:     cell = "LOST AT THE HANDOFF (present at </think> but the answer does not read it)"
elif (not dec_boundary) and reads_all:   cell = "NONLINEAR / LATE (read exists but value not linearly at the boundary)"
else:                                    cell = "NEVER CONSOLIDATED (not present and not read)"

L = []
L.append("THINK -> ANSWER INFORMATION TRANSMISSION  (DeepSeek-R1-Distill-Qwen-1.5B)")
L.append("=" * 72)
L.append(f"examples={N}  seq_len={ref_len}  value-classes={len(set(y))}  chance={chance:.3f}")
L.append(f"clean top-1 == secret value: {top1:.1%}    logit-diff clean={cl:.2f}  corrupt={float(corr_ld.mean()):.2f}")
L.append("")
L.append("SENSE 2 - representational (logit lens + probes):")
L.append(f"  peak probe acc   value_think={probe['value_think'].max():.2f}  boundary={probe['boundary'].max():.2f}  answer={probe['answer'].max():.2f}")
L.append(f"  decodable at handoff(boundary)={dec_boundary}   at answer position={dec_answer}")
L.append(f"  boundary->answer peak-accuracy drop = {b_ans_drop:+.2f}  (large positive = representational attenuation across the boundary)")
L.append("")
L.append("SENSE 3 - routing (attention knockout, block ALL layers, logit-diff drop of clean {:.2f}):".format(cl))
for t in KO_TARGETS:
    L.append(f"  answer -/-> {t:13s}: {ko_total[t]['ldrop']:.2f}  (KL={ko_total[t]['kl']:.2f})")
L.append(f"  handoff funnels through the </think> token more than raw content: {via_boundary}")
L.append("")
L.append("2x2 VERDICT: " + cell)
L.append("")
L.append("Caveat: single-layer knockout is a direct-edge test; multi-hop routing")
L.append("(think -> intermediate positions -> answer) is only fully severed by the block-all-layers result.")
text = "\n".join(L)
print(text)

with open(f"{OUT_DIR}/analysis.txt", "w") as f:
    f.write(text)
with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump(dict(
        n_examples=N, seq_len=int(ref_len), n_value_classes=len(set(y)), chance=chance,
        clean_top1=float(top1), clean_logit_diff=cl, corrupt_logit_diff=float(corr_ld.mean()),
        probe_peak={k: float(probe[k].max()) for k in LOCI},
        lens_min_rank={k: float(lens_rank[k].min()) for k in LOCI},
        boundary_to_answer_probe_drop=b_ans_drop,
        knockout_block_all={k: ko_total[k] for k in KO_TARGETS},
        probe_by_layer={k: probe[k].tolist() for k in LOCI},
        lens_rank_by_layer={k: lens_rank[k].tolist() for k in LOCI},
        knockout_ldrop_by_layer={k: ko_ldrop[k].tolist() for k in KO_TARGETS},
        verdict=cell,
    ), f, indent=2)
print("\nsaved: outputs/summary.png, outputs/analysis.txt, outputs/metrics.json")